# 03 Q1 runs

Switches off every planned group (preregistration 4.1, 4.2, 6, 8.1-8.3) on the evaluation prompts and saves one CSV per run in `results/q1_runs/runs/`. About 8 GPU hours: run as **Save Version, Save & Run All**. Runs already saved are skipped, so a new session resumes where the last one stopped.

Setup: Accelerator GPU T4, Internet on, secret `HF_TOKEN`.

In [ ]:
# 1. code and libraries
!rm -rf helix_addition && git clone https://github.com/amark-23/helix_addition.git
!pip install -q git+https://github.com/safety-research/circuit-tracer@v0.5.3 pytest

In [ ]:
# 2. make helixadd importable and log in
import os, sys
os.chdir("/kaggle/working/helix_addition")          # the package reads results/ relative to the repo
sys.path.insert(0, "/kaggle/working/helix_addition/src")

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(UserSecretsClient().get_secret("HF_TOKEN"))

In [ ]:
# 3. tests (tokenization, labels, groups, plan)
!python -m pytest -q

In [ ]:
# 4. model
from helixadd import model as hmodel, q1, intervene, prompts
model = hmodel.load_model()

## Sanity check: the clean run reproduces the clean accuracy

No group is switched off here, so this shows nothing about Q1.

In [ ]:
runs = q1.plan()
pairs = q1.eval_pairs()
watch = {n: intervene.by_layer(runs[n]["layer"], runs[n]["feature"]) for n in q1.WATCHED}
clean = q1.run_one(model, runs["clean"], pairs[:128], watch=watch)
print("clean accuracy on 128 evaluation prompts:", (clean["outcome"] == "N").mean())
print(clean[[f"S {n}" for n in q1.WATCHED]].describe().round(2))
assert (clean["outcome"] == "N").mean() > 0.98

## All runs

In [ ]:
q1.run_all(model, runs, pairs)

In [ ]:
# one file to download
!zip -qr q1_runs.zip results/q1_runs && ls -la q1_runs.zip